# Hyperparameter Search — Validation Set Only

Selects the XGBoost and LightGBM configurations used by `03_xgboost_sybil`, `04_lightgbm_sybil`, and the
robustness notebooks (`07`, `08`, `09`). The model settings that are not tuned come from `sp.xgb_model` and
`sp.lgbm_model`, the same definitions the model notebooks use.

**Protocol**
- Split: the stratified **group** split (`sp.make_splits(..., method='group')`), the revision's
  primary evaluation. The same selected configuration is reused in `11_split_comparison`.
- The test partition is removed before any model is fit; this notebook never sees test labels.
- Each configuration is trained once (seed 42) on the upsampled training partition, with early
  stopping on validation log-loss (patience 50, ceiling 5,000 rounds).
- **Selection criterion**: validation F1 at the F1-maximising validation threshold. Ties are broken
  by validation average precision (AP).

**Search space** (full grids over the ranges stated in the paper)

| Model | Stage | Grid | Configs |
|---|---|---|---|
| XGBoost | 1 | `learning_rate` {0.3, 0.1, 0.05} × `max_depth` {6, 8, 10, 12} × `subsample` {1.0, 0.7, 0.5} × `colsample_bytree` {1.0, 0.8} | 72 |
| XGBoost | 2 | `min_child_weight` {1, 5, 10} at the stage-1 best | +2 |
| LightGBM | — | `num_leaves` {31, 63, 127, 255, 511, 1023} × `learning_rate` {0.1, 0.05} × `subsample` {1.0, 0.7} (`subsample_freq=1`) × `reg_lambda` {0, 1}; `colsample_bytree` 0.8 | 48 |

Results are written after every configuration to `results/02_search_<model>.csv`, so an
interrupted run resumes where it stopped. The selected configurations go to
`results/02_hyperparameter_search.json`, which the model notebooks read.

In [1]:
import os, time, itertools, warnings
import pandas as pd
from sklearn.metrics import f1_score, average_precision_score, log_loss
import lightgbm as lgb_lib
import sybil_pipeline as sp
warnings.filterwarnings('ignore')

DATA_DIR = './data'
RESULTS_DIR = './results'
SEED = sp.SEED
os.makedirs(RESULTS_DIR, exist_ok=True)

df, _, _ = sp.build_master_df(DATA_DIR)
S = sp.make_splits(df, sp.FEATS, method='group', seed=SEED)   # selection is done on the group split
X_train, y_train = S['X_train'], S['y_train']
X_val,   y_val   = S['X_val'],   S['y_val']
for k in ['X_test', 'y_test', 'idx_test']:
    del S[k]                    # test partition is not available in this notebook
print(sp.split_summary(df, {**S, 'idx_test': []}).iloc[:2].to_string(index=False))

[1] L0 features: 434,786 addresses, 55 cols


[2] Gas provision: 604,365 edges before 2024-05-02 00:00:00 UTC (296 later edges dropped)


[3] Labeled addresses in provision network: 4,308


[4] Provider and tree features: 88 cols


[5] CEX/DEX merged: 90 cols


[6] Labels: Sybil=18,211 (4.19%)


[7] Chain features, taxonomy, gas provision trees (371,764 trees)


All 62 features present ✓  (76.0s)


     Split  Total  Sybil  NonSybil  SybilPct
     Train 213046   8924    204122      4.19
Validation  91305   3824     87481      4.19


In [2]:
def val_scores(probs):
    thr = sp.best_f1_threshold(y_val, probs)
    return dict(val_f1=f1_score(y_val, (probs >= thr).astype(int)),
                val_ap=average_precision_score(y_val, probs),
                val_logloss=log_loss(y_val, probs), val_threshold=thr)

def run_grid(name, configs, fit):
    """Fit every config not already in results/02_search_<name>.csv; return the full table."""
    path = os.path.join(RESULTS_DIR, f'02_search_{name}.csv')
    done = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
    keys = list(configs[0])
    seen = set(map(tuple, done[keys].astype(str).values)) if len(done) else set()
    for i, cfg in enumerate(configs, 1):
        if tuple(str(cfg[k]) for k in keys) in seen:
            continue
        t0 = time.time()
        probs, rounds = fit(cfg)
        row = {**cfg, **val_scores(probs), 'rounds': rounds, 'seconds': round(time.time() - t0, 1)}
        done = pd.concat([done, pd.DataFrame([row])], ignore_index=True)
        done.to_csv(path, index=False)
        print(f"[{i}/{len(configs)}] {cfg}  val_F1={row['val_f1']:.4f}  AP={row['val_ap']:.4f}  "
              f"rounds={rounds}  {row['seconds']}s", flush=True)
    for k in keys:                          # restore dtypes after CSV round-trip
        done[k] = done[k].astype(type(configs[0][k]))
    return done

def best(table):
    return table.sort_values(['val_f1', 'val_ap'], ascending=False).iloc[0]

INT_PARAMS = {'max_depth', 'min_child_weight', 'num_leaves'}

def as_params(row, keys):
    # results-table row -> hyperparameter dict with the right Python types
    return {k: int(row[k]) if k in INT_PARAMS else float(row[k]) for k in keys}

## XGBoost — stage 1

In [3]:
def fit_xgb(cfg):
    m = sp.xgb_model(cfg, SEED)   # fixed settings as in the model notebooks
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return m.predict_proba(X_val)[:, 1], int(m.best_iteration)

xgb_grid = [dict(learning_rate=lr, max_depth=d, subsample=s, colsample_bytree=c, min_child_weight=1)
            for lr, d, s, c in itertools.product([0.3, 0.1, 0.05], [6, 8, 10, 12],
                                                 [1.0, 0.7, 0.5], [1.0, 0.8])]
xgb1 = run_grid('xgboost', xgb_grid, fit_xgb)
print(xgb1.sort_values('val_f1', ascending=False).head(10).round(4).to_string(index=False))

[1/72] {'learning_rate': 0.3, 'max_depth': 6, 'subsample': 1.0, 'colsample_bytree': 1.0, 'min_child_weight': 1}  val_F1=0.6851  AP=0.7369  rounds=646  33.3s


[2/72] {'learning_rate': 0.3, 'max_depth': 6, 'subsample': 1.0, 'colsample_bytree': 0.8, 'min_child_weight': 1}  val_F1=0.6886  AP=0.7387  rounds=779  34.9s


[3/72] {'learning_rate': 0.3, 'max_depth': 6, 'subsample': 0.7, 'colsample_bytree': 1.0, 'min_child_weight': 1}  val_F1=0.6891  AP=0.7349  rounds=556  27.5s


[4/72] {'learning_rate': 0.3, 'max_depth': 6, 'subsample': 0.7, 'colsample_bytree': 0.8, 'min_child_weight': 1}  val_F1=0.6873  AP=0.7364  rounds=659  31.1s


[5/72] {'learning_rate': 0.3, 'max_depth': 6, 'subsample': 0.5, 'colsample_bytree': 1.0, 'min_child_weight': 1}  val_F1=0.6870  AP=0.7328  rounds=565  32.0s


...

[69/72] {'learning_rate': 0.05, 'max_depth': 12, 'subsample': 0.7, 'colsample_bytree': 1.0, 'min_child_weight': 1}  val_F1=0.6963  AP=0.7543  rounds=562  55.4s


[70/72] {'learning_rate': 0.05, 'max_depth': 12, 'subsample': 0.7, 'colsample_bytree': 0.8, 'min_child_weight': 1}  val_F1=0.6982  AP=0.7555  rounds=580  57.9s


[71/72] {'learning_rate': 0.05, 'max_depth': 12, 'subsample': 0.5, 'colsample_bytree': 1.0, 'min_child_weight': 1}  val_F1=0.7000  AP=0.7528  rounds=521  53.4s


[72/72] {'learning_rate': 0.05, 'max_depth': 12, 'subsample': 0.5, 'colsample_bytree': 0.8, 'min_child_weight': 1}  val_F1=0.7018  AP=0.7538  rounds=547  55.6s


 learning_rate  max_depth  subsample  colsample_bytree  min_child_weight  val_f1  val_ap  val_logloss  val_threshold  rounds  seconds
          0.05         12        0.5               0.8                 1  0.7018  0.7538       0.0807         0.5488     547     55.6
          0.05         10        0.7               1.0                 1  0.7003  0.7510       0.0835         0.5817     911     69.3
          0.10         10        0.5               0.8                 1  0.7000  0.7552       0.0811         0.4889     444     38.4
          0.05         12        0.5               1.0                 1  0.7000  0.7528       0.0813         0.6483     521     53.4
          0.05         10        0.7               0.8                 1  0.6995  0.7545       0.0823         0.5531     925     68.0
          0.10         10        0.7               1.0                 1  0.6993  0.7502       0.0844         0.4959     452     35.4
          0.10         12        0.7               0.8        

## XGBoost — stage 2 (`min_child_weight`)

In [4]:
b1 = best(xgb1)
xgb_grid2 = xgb_grid + [{**as_params(b1, ['learning_rate', 'max_depth', 'subsample', 'colsample_bytree']),
                         'min_child_weight': w} for w in [5, 10]]
xgb_all = run_grid('xgboost', xgb_grid2, fit_xgb)
xgb_best = best(xgb_all)
XGB_SELECTED = as_params(xgb_best, ['learning_rate', 'max_depth', 'subsample',
                                    'colsample_bytree', 'min_child_weight'])
print('Selected XGBoost config:', XGB_SELECTED)
print(xgb_best[['val_f1', 'val_ap', 'rounds']].to_string())

[73/74] {'learning_rate': 0.05, 'max_depth': 12, 'subsample': 0.5, 'colsample_bytree': 0.8, 'min_child_weight': 5}  val_F1=0.6999  AP=0.7540  rounds=878  76.2s


[74/74] {'learning_rate': 0.05, 'max_depth': 12, 'subsample': 0.5, 'colsample_bytree': 0.8, 'min_child_weight': 10}  val_F1=0.7024  AP=0.7534  rounds=1214  87.4s


Selected XGBoost config: {'learning_rate': 0.05, 'max_depth': 12, 'subsample': 0.5, 'colsample_bytree': 0.8, 'min_child_weight': 10}
val_f1       0.702416
val_ap       0.753370
rounds    1214.000000


## LightGBM

In [5]:
def fit_lgbm(cfg):
    m = sp.lgbm_model(cfg, SEED)   # fixed settings as in the model notebooks
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)],
          callbacks=[lgb_lib.early_stopping(50, verbose=False), lgb_lib.log_evaluation(-1)])
    return m.predict_proba(X_val)[:, 1], int(m.best_iteration_)

lgbm_grid = [dict(num_leaves=nl, learning_rate=lr, subsample=s, reg_lambda=l2)
             for nl, lr, s, l2 in itertools.product([31, 63, 127, 255, 511, 1023], [0.1, 0.05],
                                                    [1.0, 0.7], [0.0, 1.0])]
lgbm_all = run_grid('lightgbm', lgbm_grid, fit_lgbm)
lgbm_best = best(lgbm_all)
LGBM_SELECTED = as_params(lgbm_best, ['num_leaves', 'learning_rate', 'subsample', 'reg_lambda'])
print('Selected LightGBM config:', LGBM_SELECTED)
print(lgbm_all.sort_values('val_f1', ascending=False).head(10).round(4).to_string(index=False))

[1/48] {'num_leaves': 31, 'learning_rate': 0.1, 'subsample': 1.0, 'reg_lambda': 0.0}  val_F1=0.6886  AP=0.7386  rounds=2084  57.8s


[2/48] {'num_leaves': 31, 'learning_rate': 0.1, 'subsample': 1.0, 'reg_lambda': 1.0}  val_F1=0.7021  AP=0.7524  rounds=2789  75.4s


[3/48] {'num_leaves': 31, 'learning_rate': 0.1, 'subsample': 0.7, 'reg_lambda': 0.0}  val_F1=0.6996  AP=0.7477  rounds=2170  66.8s


[4/48] {'num_leaves': 31, 'learning_rate': 0.1, 'subsample': 0.7, 'reg_lambda': 1.0}  val_F1=0.7014  AP=0.7530  rounds=2408  73.4s


[5/48] {'num_leaves': 31, 'learning_rate': 0.05, 'subsample': 1.0, 'reg_lambda': 0.0}  val_F1=0.6933  AP=0.7465  rounds=4008  107.2s


...

[45/48] {'num_leaves': 1023, 'learning_rate': 0.05, 'subsample': 1.0, 'reg_lambda': 0.0}  val_F1=0.7011  AP=0.7520  rounds=132  39.1s


[46/48] {'num_leaves': 1023, 'learning_rate': 0.05, 'subsample': 1.0, 'reg_lambda': 1.0}  val_F1=0.7011  AP=0.7574  rounds=173  46.3s


[47/48] {'num_leaves': 1023, 'learning_rate': 0.05, 'subsample': 0.7, 'reg_lambda': 0.0}  val_F1=0.6998  AP=0.7563  rounds=141  40.9s


[48/48] {'num_leaves': 1023, 'learning_rate': 0.05, 'subsample': 0.7, 'reg_lambda': 1.0}  val_F1=0.7030  AP=0.7593  rounds=164  44.4s


Selected LightGBM config: {'num_leaves': 511, 'learning_rate': 0.05, 'subsample': 1.0, 'reg_lambda': 1.0}
 num_leaves  learning_rate  subsample  reg_lambda  val_f1  val_ap  val_logloss  val_threshold  rounds  seconds
        511           0.05        1.0         1.0  0.7046  0.7561       0.0787         0.6157     315     48.1
        255           0.05        0.7         1.0  0.7046  0.7570       0.0789         0.5449     608     56.3
         63           0.10        0.7         1.0  0.7044  0.7523       0.0823         0.6201    1178     50.1
        511           0.05        0.7         1.0  0.7041  0.7579       0.0780         0.6091     301     49.9
       1023           0.05        0.7         1.0  0.7030  0.7593       0.0773         0.5338     164     44.4
         63           0.05        0.7         0.0  0.7029  0.7511       0.0821         0.6006    2089     83.2
         31           0.05        0.7         1.0  0.7026  0.7552       0.0819         0.6981    4625    133.3
      

## Marginal effect of each hyperparameter (validation F1)

For each hyperparameter value: the best validation F1 over all configs with that value. This is
the ablation view of the grid.

In [6]:
def marginal(table, params):
    out = []
    for p in params:
        g = table.groupby(p)['val_f1'].max()
        out += [dict(param=p, value=v, best_val_f1=round(f, 4)) for v, f in g.items()]
    return pd.DataFrame(out)

print('XGBoost (stage 1 grid):')
print(marginal(xgb1, ['learning_rate', 'max_depth', 'subsample', 'colsample_bytree']).to_string(index=False))
print('\nLightGBM:')
print(marginal(lgbm_all, ['num_leaves', 'learning_rate', 'subsample', 'reg_lambda']).to_string(index=False))

XGBoost (stage 1 grid):
           param  value  best_val_f1
   learning_rate   0.05       0.7018
   learning_rate   0.10       0.7000
   learning_rate   0.30       0.6973
       max_depth   6.00       0.6975
       max_depth   8.00       0.6973
       max_depth  10.00       0.7003
       max_depth  12.00       0.7018
       subsample   0.50       0.7018
       subsample   0.70       0.7003
       subsample   1.00       0.6984
colsample_bytree   0.80       0.7018
colsample_bytree   1.00       0.7003

LightGBM:
        param   value  best_val_f1
   num_leaves   31.00       0.7026
   num_leaves   63.00       0.7044
   num_leaves  127.00       0.7015
   num_leaves  255.00       0.7046
   num_leaves  511.00       0.7046
   num_leaves 1023.00       0.7030
learning_rate    0.05       0.7046
learning_rate    0.10       0.7044
    subsample    0.70       0.7046
    subsample    1.00       0.7046
   reg_lambda    0.00       0.7029
   reg_lambda    1.00       0.7046


## Save selected configurations

In [7]:
sp.save_results([], '02_hyperparameter_search', extra=dict(
    selection='max validation F1 at validation-optimal threshold; ties by validation AP; seed 42',
    xgb_selected=XGB_SELECTED, lgbm_selected=LGBM_SELECTED,
    xgb_selected_val=dict(val_f1=float(xgb_best.val_f1), val_ap=float(xgb_best.val_ap), rounds=int(xgb_best.rounds)),
    lgbm_selected_val=dict(val_f1=float(lgbm_best.val_f1), val_ap=float(lgbm_best.val_ap), rounds=int(lgbm_best.rounds)),
    n_configs=dict(xgboost=len(xgb_all), lightgbm=len(lgbm_all)),
))

Saved results/02_hyperparameter_search.json
